# 04 - Kaggle Training Notebook (Cloud)

Notebook này chạy **hoàn toàn trên Kaggle** để train model và đóng gói artifacts cho backend local.

## Cách dùng nhanh

1. Vào `https://kaggle.com` -> New Notebook.
2. Add dataset chứa `DataCoSupplyChainDataset.csv`.
3. Copy toàn bộ notebook này vào Kaggle.
4. Run All (~3-5 phút).
5. Vào tab **Output** -> download `ml_artifacts.zip`.
6. Giải nén vào `backend/ml/` trong project local.

## CELL 1: Setup

In [ ]:
import os
import glob
import json
import zipfile
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working/ml_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Setup done — Output:', OUTPUT_DIR)

## CELL 2: Load Data

In [ ]:
# Scan all csv files in Kaggle input
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
print('Files found:')
for f in csv_files:
    print(' -', f)

# Prefer exact dataset file name
exact_match = [f for f in csv_files if f.endswith('DataCoSupplyChainDataset.csv')]
fallback = [f for f in csv_files if 'DataCo' in os.path.basename(f)]

if exact_match:
    dataset_path = exact_match[0]
elif fallback:
    dataset_path = fallback[0]
else:
    raise FileNotFoundError('Cannot find DataCo dataset CSV in /kaggle/input')

print()
print(f'Loading: {dataset_path}')
df = pd.read_csv(dataset_path, encoding='cp1252')
print(f'Shape: {df.shape}')
print('Target distribution:')
print(df['Late_delivery_risk'].value_counts(normalize=True))

## CELL 3: Feature Engineering

In [ ]:
ALLOWED_COLS = [
    'Shipping Mode',
    'Days for shipment (scheduled)',
    'Market',
    'Order Region',
    'Category Name',
    'Customer Segment',
    'Department Name',
    'order date (DateOrders)',
    'Benefit per order',
    'Order Item Discount Rate',
]
TARGET = 'Late_delivery_risk'

df_model = df[ALLOWED_COLS + [TARGET]].copy()

# Datetime-derived features
df_model['order date (DateOrders)'] = pd.to_datetime(
    df_model['order date (DateOrders)'], errors='coerce'
)
df_model['Month'] = df_model['order date (DateOrders)'].dt.month
df_model['Day_of_Week'] = df_model['order date (DateOrders)'].dt.dayofweek
df_model['Quarter'] = df_model['order date (DateOrders)'].dt.quarter
df_model['Is_Peak_Season'] = df_model['Month'].isin([10, 11, 12]).astype(int)
df_model.drop(columns=['order date (DateOrders)'], inplace=True)

# Missing handling
for col in df_model.select_dtypes(include='number').columns:
    if col != TARGET:
        df_model[col].fillna(df_model[col].median(), inplace=True)
for col in df_model.select_dtypes(include='object').columns:
    df_model[col].fillna('Unknown', inplace=True)

print('✅ Feature engineering done')
print('Missing after fill:', int(df_model.isnull().sum().sum()))

## CELL 4: Encode & Save encoding_map

In [ ]:
CAT_COLS = [
    'Shipping Mode',
    'Market',
    'Order Region',
    'Category Name',
    'Customer Segment',
    'Department Name',
]

encoding_map = {}

for col in CAT_COLS:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))
    encoding_map[col] = {str(cls): int(idx) for idx, cls in enumerate(le.classes_)}
    df_model.drop(columns=[col], inplace=True)

with open(f'{OUTPUT_DIR}/encoding_map.json', 'w') as f:
    json.dump(encoding_map, f, indent=2)

print('✅ encoding_map.json saved')
print('   Keys:', list(encoding_map.keys()))

## CELL 5: Define Features & Split

In [ ]:
FINAL_FEATURES = [
    'Shipping Mode_enc',
    'Days for shipment (scheduled)',
    'Market_enc',
    'Order Region_enc',
    'Category Name_enc',
    'Customer Segment_enc',
    'Department Name_enc',
    'Month',
    'Day_of_Week',
    'Is_Peak_Season',
    'Quarter',
    'Benefit per order',
    'Order Item Discount Rate',
]

X = df_model[FINAL_FEATURES].copy()
y = df_model[TARGET].copy()

with open(f'{OUTPUT_DIR}/feature_names.json', 'w') as f:
    json.dump(FINAL_FEATURES, f, indent=2)

print(f'✅ feature_names.json saved — {len(FINAL_FEATURES)} features')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## CELL 6: Scale & Save scaler

In [ ]:
NUM_COLS = [
    'Days for shipment (scheduled)',
    'Benefit per order',
    'Order Item Discount Rate',
]

scaler = StandardScaler()
X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

joblib.dump(scaler, f'{OUTPUT_DIR}/scaler.pkl')
print('✅ scaler.pkl saved')

## CELL 7: Train Models

In [ ]:
print('Training Logistic Regression (baseline)...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

print('Training Random Forest (final model)...')
rf = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
print('✅ Models trained')

## CELL 8: Evaluate với threshold=0.3

In [ ]:
THRESHOLD = 0.3

for name, model in [('Logistic Regression', lr), ('Random Forest', rf)]:
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= THRESHOLD).astype(int)
    auc = roc_auc_score(y_test, prob)

    print(f"\n{'='*40}")
    print(f'  {name} (threshold={THRESHOLD})')
    print(f"{'='*40}")
    print(classification_report(y_test, pred, target_names=['On Time', 'Late']))
    print(f'  AUC-ROC: {auc:.4f}')

# Confusion matrix for RF
fig, ax = plt.subplots(figsize=(6, 5))
rf_pred = (rf.predict_proba(X_test)[:, 1] >= THRESHOLD).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    display_labels=['On Time', 'Late'],
    ax=ax,
)
ax.set_title(f'Random Forest — threshold={THRESHOLD}')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=100)
plt.show()

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=FINAL_FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['crimson'] * 5 + ['steelblue'] * (len(FINAL_FEATURES) - 5)
feat_imp.plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_title('Feature Importances — Random Forest')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_importance.png', dpi=100)
plt.show()

## CELL 9: Export Model + Config

In [ ]:
joblib.dump(rf, f'{OUTPUT_DIR}/model.pkl')
print('✅ model.pkl saved')

model_config = {
    'model_type': 'RandomForestClassifier',
    'threshold': THRESHOLD,
    'n_features': len(FINAL_FEATURES),
    'feature_names': FINAL_FEATURES,
    'num_cols_to_scale': NUM_COLS,
    'target': 'Late_delivery_risk',
    'class_labels': ['On Time', 'Late'],
    'trained_on': '180519 rows',
}

with open(f'{OUTPUT_DIR}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print('✅ model_config.json saved')

## CELL 10: Zip tất cả để download 1 lần

In [ ]:
zip_path = '/kaggle/working/ml_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zipf.write(fpath, arcname=fname)
            print(f'  Added: {fname}')

print(f'\n✅ All artifacts zipped: {zip_path}')
print('\n📥 NEXT STEPS:')
print('  1. Kaggle -> tab Output -> Download ml_artifacts.zip')
print('  2. Giải nén vào backend/ml/')
print('  3. Verify: ls backend/ml/')

## CELL 11: Sanity Check trước khi download

In [ ]:
loaded_model = joblib.load(f'{OUTPUT_DIR}/model.pkl')
loaded_scaler = joblib.load(f'{OUTPUT_DIR}/scaler.pkl')

with open(f'{OUTPUT_DIR}/encoding_map.json') as f:
    loaded_map = json.load(f)
with open(f'{OUTPUT_DIR}/feature_names.json') as f:
    loaded_features = json.load(f)

sample = X_test.iloc[:1].copy()
prob = loaded_model.predict_proba(sample)[0][1]
pred = int(prob >= THRESHOLD)

print('✅ Sanity check passed')
print(f'   Model type : {type(loaded_model).__name__}')
print(f'   Scaler type: {type(loaded_scaler).__name__}')
print(f'   N features : {len(loaded_features)}')
print(f'   Encoding   : {list(loaded_map.keys())}')
print(f'   Sample pred: prob={prob:.3f}, label={"Late" if pred else "On Time"}')
print('\n🎉 Ready to download!')

## Sau khi download từ Kaggle

1. Download `ml_artifacts.zip` từ tab **Output**.
2. Giải nén toàn bộ file vào thư mục local `backend/ml/`.
3. Kiểm tra đủ 5 artifacts:
   - `model.pkl`
   - `scaler.pkl`
   - `encoding_map.json`
   - `feature_names.json`
   - `model_config.json`
4. Sau đó mới tiếp tục Prompt 5 (FastAPI backend).